# TRAFFIC TREND — ROAD SAFETY INTELLIGENCE PLATFORM
## Exploratory Data Analysis & Statistical Modeling
**Research Focus:** Multi-Year Road Crash Severity, Spatial Hotspotting, and Temporal Diurnal Patterns
---


### 1. Environment Setup & Data Ingestion
Loads the verified benchmark demonstration dataset representing multi-year recorded road crashes.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Load verified demonstration dataset
data_path = Path('../data/raw/traffic_accidents_benchmark_demonstration.csv')
df = pd.read_csv(data_path)
print(f'Successfully loaded dataset: {df.shape[0]:,} records, {df.shape[1]} columns.')
df.head(3)

### 2. Schema Profiling & Data Quality Audit
Evaluates completeness, duplicate records, and data types across canonical dimensions.

In [2]:
print('--- Dataset Summary Info ---')
print(df.info())
print('\n--- Missing Values Breakdown ---')
missing = df.isna().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'Zero missing values identified.')
print(f'\nExact Duplicate Rows: {df.duplicated().sum()}')

### 3. Executive KPIs & Casualty Computation
Strictly distinguishing incident frequency (crashes) from human casualty figures (fatalities and injuries).

In [3]:
total_crashes = len(df)
total_fatalities = df['fatalities_count'].sum()
total_injured = df['injured_count'].sum()
total_casualties = total_fatalities + total_injured
fatality_rate = (total_fatalities / total_crashes) * 100.0
casualty_rate = total_casualties / total_crashes

print('=== EXECUTIVE KPI SUMMARY ===')
print(f'Total Recorded Collisions: {total_crashes:,}')
print(f'Total Fatalities:          {total_fatalities:,}')
print(f'Total Injured Persons:     {total_injured:,}')
print(f'Total Casualties:          {total_casualties:,}')
print(f'Fatality Rate per 100:     {fatality_rate:.2f}%')
print(f'Casualties per Crash:      {casualty_rate:.2f}')

### 4. Severity Distribution & Vehicle Vulnerability
Investigates which primary vehicle types suffer disproportionately fatal impact ratios.

In [4]:
vehicle_summary = df.groupby('primary_vehicle_type').agg(
    crashes=('accident_id', 'count'),
    fatalities=('fatalities_count', 'sum'),
    injuries=('injured_count', 'sum')
).reset_index()
vehicle_summary['fatality_rate_pct'] = (vehicle_summary['fatalities'] / vehicle_summary['crashes']) * 100.0
vehicle_summary.sort_values('crashes', ascending=False)

### 5. Temporal Diurnal Patterns & Rush-Hour Dynamics
Analyzes hourly crash occurrences across the 24-hour cycle to discover peak risk windows.

In [5]:
hourly = df.groupby('hour_of_day')['accident_id'].count().reset_index(name='crashes')
peak_threshold = hourly['crashes'].quantile(0.75)
peak_hours = hourly[hourly['crashes'] >= peak_threshold]['hour_of_day'].tolist()
print(f'Identified Peak Diurnal Crash Hours (Top 25%): {peak_hours}')

### 6. Statistical Anomaly & Outlier Detection
Applies parametric Z-scores to isolate jurisdictions with statistically anomalous accident spikes.

In [6]:
state_counts = df['state'].value_counts().reset_index()
state_counts.columns = ['state', 'crashes']
mean_vol = state_counts['crashes'].mean()
std_vol = state_counts['crashes'].std()
state_counts['z_score'] = ((state_counts['crashes'] - mean_vol) / std_vol).round(2)
anomalous_states = state_counts[state_counts['z_score'] > 1.96]
print(f'States Exceeding Statistical Norms (Z > 1.96):\n{anomalous_states}')

### 7. Analytical Limitations & Academic Conclusion
- **Exposure Limitation:** High aggregate crash volume reflects raw recorded collisions, not per-capita risk, unless normalized by vehicle-miles traveled (VMT).
- **Causality Warning:** Correlation with weather or vehicle classes does not prove exclusive causation.
- **Integration:** These analytical definitions correspond 1:1 with the FastAPI backend and Next.js frontend.